# NASA Li-Ion battery prediction task - physical SYNE-KANs and MLP baseline 

This notebook implements the NASA battery regression/prediction tasj in *Learning Nonlinear Heterogeneity in Physical Kolmogorov-Arnold Networks*.

It compares a single `[9, 12, 1]` physical KAN, with 12 SYNEs per synapse, against two ReLU MLP baselines of size (`[9,50,50,1]` and `[9,300,300,300,300,300,1]`), which are the smallest MLP considered in the paper (3101 parameters, the SYNE-KAN is 7213 parameters) and the largest MLP. 

## Before running

Place `SYNE_device_digital_twin.pt` beside the notebook, or set `SYNE_TWIN_PATH` to its location.

Download the CSV dataset from https://www.kaggle.com/datasets/patrickfleith/nasa-battery-dataset (the original NASA source is https://data.nasa.gov/dataset/li-ion-battery-aging-datasets). Point `NASA_DATA_ROOT` at the directory holding `metadata.csv` and the `data` folder.

In [ ]:
from __future__ import annotations

import copy
import gc
import hashlib
import json
import math
import os
import platform
import random
import re
import sys
import time
import warnings
from contextlib import nullcontext
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str((Path.cwd() / ".matplotlib-cache").resolve()))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = bool(DEVICE.type == "cuda")
if torch.cuda.is_available():
    torch.set_float32_matmul_precision("high")


PLOT_DPI = 240
KAN_COLOUR = "#0b3c6f"
MLP_COLOURS = {"d2_w50": "#f39c12", "d5_w300": "#c65d00"}
TARGET_COLOUR = "#202020"
GRID_ALPHA = 0.22

plt.rcParams.update(
    {
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.titlesize": 11,
        "axes.labelsize": 10,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "legend.fontsize": 9,
    }
)


def save_figure(fig, path, dpi=PLOT_DPI):
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")


def seed_everything(seed: int) -> None:
    '''Seed model initialisation, data ordering and NumPy sampling.'''
    random.seed(int(seed))
    np.random.seed(int(seed))
    torch.manual_seed(int(seed))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(seed))
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def autocast_context(enabled: bool):
    # Keep compatibility with older PyTorch versions.
    return torch.cuda.amp.autocast() if enabled else nullcontext()


def make_grad_scaler(enabled: bool):
    return torch.cuda.amp.GradScaler(enabled=bool(enabled))


def sha256_file(path, block_size: int = 1 << 20) -> str:
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        chunk = handle.read(block_size)
        while chunk:
            digest.update(chunk)
            chunk = handle.read(block_size)
    return digest.hexdigest()


def recipe_signature(hp: dict) -> str:
    payload = json.dumps(hp, sort_keys=True, separators=(",", ":"), default=str)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:12]


VERSIONS = {
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "device": str(DEVICE),
}
print(f"Python {VERSIONS['python']} | PyTorch {torch.__version__} | device: {DEVICE} | AMP: {USE_AMP}")

## Run configuration

The nine-input snapshot table is built once from every discharge cycle, then split per seed into a 72% training, 8% validation and 20% test partition. Test nMSE is the mean squared error after min-max scaling the target to `[-1, 1]`.

The KAN recipe is fixed. Each MLP baseline uses its own fixed optimiser recipe.

In [ ]:
RUN_PHYSICAL_KAN = True         # [9,12,1] physical SYNE-KAN, 12 SYNEs/synapse
RUN_MLP_BASELINES = True        # the two ReLU MLP baselines

PROJECT_ROOT = Path.cwd()
TWIN_PATH = Path(
    os.environ.get("SYNE_TWIN_PATH", PROJECT_ROOT / "SYNE_device_digital_twin.pt")
).expanduser()
DATA_ROOT = Path(
    os.environ.get("NASA_DATA_ROOT", PROJECT_ROOT / "cleaned_dataset")
).expanduser()
RESULTS_ROOT = Path(
    os.environ.get("NASA_RESULTS_DIR", PROJECT_ROOT / "results_nasa_release_v3")
).expanduser()

# The nine-input table is deterministic in the sampling seed and is shared by
# every model.
SAMPLING_SEED = 0
SAMPLES_PER_CYCLE = 50
MIN_GAP_FRACTION = 0.05

SEEDS = [24460, 24461, 24462]

KAN_MAX_EPOCHS = 800
KAN_WARMUP_INITIALISATIONS = 10
KAN_WARMUP_EPOCHS = 10
KAN_EVALUATE_EVERY = 10
KAN_EARLY_STOP_MIN_EPOCH = 500
KAN_EARLY_STOP_PATIENCE = 100
KAN_EARLY_STOP_MIN_DELTA = 0.04e-3
TWIN_MICROBATCH = 131072

MLP_MAX_EPOCHS = 800
MLP_BATCH_SIZE = 2048
MLP_COSINE_T_MAX = 810

OUTPUT_ROOT = RESULTS_ROOT
PREDICTION_ROOT = OUTPUT_ROOT / "predictions"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PREDICTION_ROOT.mkdir(parents=True, exist_ok=True)

print("Digital twin:", TWIN_PATH)
print("Dataset root:", DATA_ROOT)
print("Results:", OUTPUT_ROOT)
print("Seeds:", SEEDS)


## Build the nine-input snapshot dataset

Each discharge cycle contributes `SAMPLES_PER_CYCLE` rows. The six snapshot features are `Time`, `Voltage_measured`, `Current_measured`, `Temperature_measured`, `Current_load` and `Voltage_load`; the three history features are the running means of `Voltage_measured`, `Voltage_load` and `Temperature_measured` over the rows before the sampled row. The target is the remaining discharge time in seconds. 

The processed table is cached in `OUTPUT_ROOT`, so the dataset is built once and then reloaded for later runs.

In [ ]:
FEATURE_COLUMNS = [
    "Time", "Voltage_measured", "Current_measured", "Temperature_measured",
    "Current_load", "Voltage_load",
    "mean_Voltage_measured", "mean_Voltage_load", "mean_Temperature_measured",
]
TARGET_COLUMN = "seconds_remaining"
SNAPSHOT_COLUMNS = FEATURE_COLUMNS[:6]
HISTORY_SOURCE_COLUMNS = ["Voltage_measured", "Voltage_load", "Temperature_measured"]


def locate_dataset_root(requested: Path) -> Path:
    requested = Path(requested).expanduser()
    for candidate in [requested, requested / "cleaned_dataset", PROJECT_ROOT / "FullDataset"]:
        if (candidate / "metadata.csv").is_file():
            return candidate.resolve()
    raise FileNotFoundError(
        f"Could not find metadata.csv beneath {requested}. Set NASA_DATA_ROOT to the "
        "directory holding metadata.csv and the data folder."
    )


def choose_column(columns, candidates):
    lookup = {str(column).lower(): str(column) for column in columns}
    for candidate in candidates:
        if str(candidate).lower() in lookup:
            return lookup[str(candidate).lower()]
    return None


def infer_battery_id(row, filename: str) -> str:
    explicit = choose_column(row.index, ["battery_id", "battery", "cell_id", "cell", "device_id"])
    if explicit is not None and pd.notna(row[explicit]):
        return str(row[explicit])
    stem = Path(filename).stem
    for pattern in [r"(?i)(B\d{4})", r"(?i)(battery[_-]?\d+)", r"(?i)(cell[_-]?\d+)"]:
        match = re.search(pattern, stem)
        if match:
            return match.group(1).lower()
    return re.split(r"[_-]", stem)[0].lower()


def resolve_cycle_path(root: Path, raw_name) -> Path:
    raw_path = Path(str(raw_name))
    candidates = []
    if raw_path.is_absolute():
        candidates.append(raw_path)
    candidates.extend([root / raw_path, root / "data" / raw_path, root / "data" / raw_path.name, root / raw_path.name])
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    matches = list(root.rglob(raw_path.name))
    if len(matches) == 1:
        return matches[0].resolve()
    raise FileNotFoundError(f"Could not resolve cycle file {raw_name!r} beneath {root}")


def sample_cycle_indices(n_rows, n_samples, rng, min_gap_fraction):
    lower = n_rows // 2
    eligible = np.arange(lower, n_rows, dtype=np.int64)
    if len(eligible) <= int(n_samples):
        return eligible
    upper = n_rows - 1
    minimum_gap = max(1, int(float(min_gap_fraction) * n_rows))
    candidates = np.sort(rng.randint(lower, upper + 1, size=max(int(n_samples) * 5, int(n_samples))))
    picks = []
    for candidate in candidates:
        candidate = int(candidate)
        if all(abs(candidate - previous) >= minimum_gap for previous in picks):
            picks.append(candidate)
        if len(picks) >= int(n_samples):
            break
    if len(picks) < int(n_samples):
        for candidate in np.linspace(lower, upper, int(n_samples), dtype=int):
            if int(candidate) not in picks:
                picks.append(int(candidate))
    return np.asarray(sorted(set(picks))[: int(n_samples)], dtype=np.int64)


def build_regression_table(data_root: Path) -> pd.DataFrame:
    '''Build the nine-input remaining-discharge-time table. History means use
    only rows before the sampled row and never see future values.'''
    metadata = pd.read_csv(data_root / "metadata.csv")
    filename_column = choose_column(metadata.columns, ["filename", "file_name", "file", "path"])
    type_column = choose_column(metadata.columns, ["type", "cycle_type", "operation"])
    if filename_column is None or type_column is None:
        raise ValueError("metadata.csv must contain filename and type columns")
    discharge = metadata[metadata[type_column].astype(str).str.lower().eq("discharge")].copy()
    if discharge.empty:
        raise RuntimeError("No discharge cycles were found in metadata.csv")

    print(f"Building regression table from {len(discharge):,} discharge cycles...")
    rng = np.random.RandomState(int(SAMPLING_SEED))
    rows = []
    for cycle_number, (_, metadata_row) in enumerate(discharge.iterrows()):
        raw_name = str(metadata_row[filename_column])
        battery_id = infer_battery_id(metadata_row, raw_name)
        cycle_path = resolve_cycle_path(data_root, raw_name)
        frame = pd.read_csv(cycle_path)
        if [c for c in SNAPSHOT_COLUMNS if c not in frame.columns]:
            continue
        values = frame[SNAPSHOT_COLUMNS].apply(pd.to_numeric, errors="coerce")
        values = values.loc[values.notna().all(axis=1)].reset_index(drop=True)
        if len(values) < 2:
            continue
        sampled = sample_cycle_indices(len(values), SAMPLES_PER_CYCLE, rng, MIN_GAP_FRACTION)
        end_time = float(values["Time"].iloc[-1])
        prefix_sum = values[HISTORY_SOURCE_COLUMNS].cumsum().to_numpy(np.float64)
        cycle_id = f"{battery_id}::{Path(raw_name).stem}::{cycle_number}"
        for row_index in sampled:
            row_index = int(row_index)
            snapshot = values.iloc[row_index]
            means = prefix_sum[row_index] / float(row_index + 1) if row_index == 0 else prefix_sum[row_index - 1] / float(row_index)
            item = {column: float(snapshot[column]) for column in SNAPSHOT_COLUMNS}
            item.update({f"mean_{c}": float(v) for c, v in zip(HISTORY_SOURCE_COLUMNS, means)})
            item.update({TARGET_COLUMN: end_time - float(snapshot["Time"]), "battery_id": battery_id, "cycle_id": cycle_id})
            rows.append(item)
        if (cycle_number + 1) % 250 == 0:
            print(f"  processed {cycle_number + 1:,}/{len(discharge):,} discharge cycles", flush=True)

    table = pd.DataFrame(rows)
    if table.empty:
        raise RuntimeError("No usable samples were built; inspect the dataset root.")
    if (table[TARGET_COLUMN] < 0).any():
        warnings.warn("Negative remaining-time targets found; check non-monotonic Time columns.")
    return table


DATA_ROOT = locate_dataset_root(DATA_ROOT)
DATASET_SIGNATURE = hashlib.sha256(
    json.dumps(
        {
            "metadata_sha256": sha256_file(DATA_ROOT / "metadata.csv"),
            "samples_per_cycle": SAMPLES_PER_CYCLE,
            "sampling_seed": SAMPLING_SEED,
            "min_gap_fraction": MIN_GAP_FRACTION,
            "feature_columns": FEATURE_COLUMNS,
        },
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()[:16]
DATASET_CACHE = OUTPUT_ROOT / f"dataset_{DATASET_SIGNATURE}.npz"

if DATASET_CACHE.is_file():
    cached = np.load(DATASET_CACHE, allow_pickle=True)
    X_ALL = cached["X"].astype(np.float32)
    Y_ALL = cached["y"].astype(np.float32)
    BATTERY_IDS = cached["battery_ids"]
    print("Loaded processed dataset cache:", DATASET_CACHE.name)
else:
    REGRESSION_TABLE = build_regression_table(DATA_ROOT)
    X_ALL = REGRESSION_TABLE[FEATURE_COLUMNS].to_numpy(np.float32)
    Y_ALL = REGRESSION_TABLE[[TARGET_COLUMN]].to_numpy(np.float32).ravel()
    BATTERY_IDS = REGRESSION_TABLE["battery_id"].to_numpy(str)
    np.savez_compressed(DATASET_CACHE, X=X_ALL, y=Y_ALL, battery_ids=BATTERY_IDS)
    print("Saved processed dataset cache:", DATASET_CACHE.name)

if X_ALL.ndim != 2 or X_ALL.shape[1] != 9:
    raise RuntimeError(f"Expected nine input features; observed shape {X_ALL.shape}")
if not np.isfinite(X_ALL).all() or not np.isfinite(Y_ALL).all():
    raise FloatingPointError("Dataset contains NaN or infinity")

print(json.dumps(
    {"rows": int(len(X_ALL)), "features": int(X_ALL.shape[1]),
     "batteries": int(len(np.unique(BATTERY_IDS))), "signature": DATASET_SIGNATURE},
    indent=2,
))

## Hyperparameters

Hyperparams for physical KAN and both sizes of MLP

In [ ]:
# Fixed physical KAN recipe:
KAN_RECIPE = {
    "architecture": [9, 12, 1],
    "SYNEs_per_synapse": 12,
    "range_init": "powerlaw",
    "init_dv_min": 1.2,
    "init_dv_max": 1.8,
    "invert_probability": 0.5,
    "span_jitter": 0.1,
    "pl_alpha": 0.25,
    "pl_beta": 1.75,
    "control_init_scale": 1.0,
    "output_gain_scale": 50.0,
    "optimizer": "adam",
    "lr_gobo": 1.660037773e-4,
    "lr_v": 1.660037773e-4,
    "lr_s": 8.300188865e-5,
    "beta1": 0.9166804489,
    "beta2": 0.9019533563,
    "adam_eps": 1e-8,
    "batch_size": 128,
    "grad_clip": 10.0,
    "scheduler": "cosine",
    "lr_min_frac": 0.08575523888,
}

# Fixed MLP baselines:
MLP_RECIPE_D5_W300 = {
    "label": "d5_w300",
    "architecture": [9, 300, 300, 300, 300, 300, 1],
    "depth": 5,
    "width": 300,
    "optimizer": "adam",
    "beta1": 0.9,
    "beta2": 0.999,
    "adam_eps": 1e-8,
    "lr": 3.7360527045869155e-05,
    "weight_decay": 4.658089432090407e-08,
    "gradient_accumulation_steps": 2,
    "scheduler": "cosine",
    "cosine_t_max": 810,
    "final_lr_fraction": 0.0004942976715740657,
    "dropout": 0.0,
    "grad_clip": 0.0,
    "batch_size": 2048,
}

MLP_RECIPE_D2_W50 = {
    "label": "d2_w50",
    "architecture": [9, 50, 50, 1],
    "depth": 2,
    "width": 50,
    "optimizer": "adam",
    "beta1": 0.9,
    "beta2": 0.999,
    "adam_eps": 1e-8,
    "lr": 0.0015412442470238616,
    "weight_decay": 3.1888796987188146e-07,
    "gradient_accumulation_steps": 16,
    "scheduler": "cosine",
    "cosine_t_max": 810,
    "final_lr_fraction": 0.0009015353408933473,
    "dropout": 0.0,
    "grad_clip": 0.0,
    "batch_size": 2048,
}

MLP_RECIPES = {"d5_w300": MLP_RECIPE_D5_W300, "d2_w50": MLP_RECIPE_D2_W50}

RELEASE_RECIPES = {"physical_kan": KAN_RECIPE, **MLP_RECIPES}
(OUTPUT_ROOT / "fixed_release_recipes.json").write_text(
    json.dumps(RELEASE_RECIPES, indent=2), encoding="utf-8"
)
print(json.dumps(RELEASE_RECIPES, indent=2))

## SYNE device digital twin

The loader accepts a saved `nn.Module` or a compatible `3-50-50-1` MLP state dictionary. The SYNE device digital twin is converted to float32, placed in evaluation mode and frozen. Its three inputs are the signal voltage and the two constant control voltages of a device.

In [ ]:
class MLP3x50x50x1(nn.Module):
    '''Architecture used for the supplied SYNE device digital twin state dictionary.'''

    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(3, 50), nn.ReLU(),
            nn.Linear(50, 50), nn.ReLU(),
            nn.Linear(50, 1),
        )

    def forward(self, inputs):
        return self.net(inputs)


def _strip_state_prefix(state: dict, prefix: str) -> dict:
    if state and all(str(key).startswith(prefix) for key in state):
        return {str(key)[len(prefix):]: value for key, value in state.items()}
    return state


def load_frozen_twin(path: Path) -> nn.Module:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(
            f"Digital twin not found: {path}\n"
            "Place SYNE_device_digital_twin.pt beside the notebook or set SYNE_TWIN_PATH."
        )
    try:
        payload = torch.load(path, map_location=DEVICE, weights_only=False)
    except TypeError:
        payload = torch.load(path, map_location=DEVICE)

    model = None
    state = None
    if isinstance(payload, nn.Module):
        model = payload
    elif isinstance(payload, (list, tuple)) and payload:
        if isinstance(payload[0], nn.Module):
            model = payload[0]
        elif isinstance(payload[0], dict):
            state = payload[0]
    elif isinstance(payload, dict):
        for key in ("model", "twin", "device_model"):
            if isinstance(payload.get(key), nn.Module):
                model = payload[key]
                break
        if model is None:
            state = payload.get("state_dict", payload.get("model_state_dict"))
            if state is None and payload and all(
                isinstance(k, str) and torch.is_tensor(v) for k, v in payload.items()
            ):
                state = payload

    if model is None and state is not None:
        state = _strip_state_prefix(state, "module.")
        variants = [state]
        for prefix in ("model.", "twin.", "network.", "net."):
            variant = _strip_state_prefix(state, prefix)
            if variant is not state:
                variants.append(variant)
        candidates = [MLP3x50x50x1(), nn.Sequential(
            nn.Linear(3, 50), nn.ReLU(), nn.Linear(50, 50), nn.ReLU(), nn.Linear(50, 1)
        )]
        for candidate in candidates:
            for candidate_state in variants:
                try:
                    candidate.load_state_dict(candidate_state, strict=True)
                    model = candidate
                    break
                except RuntimeError:
                    continue
            if model is not None:
                break
        if model is None:
            raise RuntimeError(f"Could not map the state dictionary in {path} to 3-50-50-1.")

    if model is None:
        raise TypeError(f"Unsupported twin payload type {type(payload).__name__} in {path}")

    model = model.to(DEVICE).float().eval()
    for parameter in model.parameters():
        parameter.requires_grad_(False)
    print(f"Loaded and froze {path.name} ({type(model).__name__})")
    return model


TWIN = load_frozen_twin(TWIN_PATH) if RUN_PHYSICAL_KAN else None

## Physical KAN

The physical SYNE-KAN places multiple instances of the SYNE digital twin on each KAN synapse. Each synapse sums `SYNEs_per_synapse` parallel SYNE responses (here 12 SYNEs/synapse). The optimised quantities are the signal ranges, the two tuning voltages per device which define the nonlinear functions, and the output gains and biases. A `[9, 12, 1]` SYNE-KAN with 12 SYNEs per synapse has a parameter count of 7,213.

In [ ]:
class PhysicalKAN(nn.Module):

    def __init__(self, twin, architecture, SYNEs_per_synapse, hp):
        super().__init__()
        self.twin = twin
        self.architecture = [int(value) for value in architecture]
        self.SYNEs_per_synapse = int(SYNEs_per_synapse)
        self.hp = dict(hp)
        self.controls_per_device = 2
        self.squash_gain = 6.0
        self.physical_epsilon = 0.02
        self.output_gain_scale = float(hp["output_gain_scale"])
        for parameter in self.twin.parameters():
            parameter.requires_grad_(False)
        self.twin.eval()

        self.S_min = nn.ParameterList()
        self.S_max = nn.ParameterList()
        self.V = nn.ParameterList()
        self.G_o = nn.ParameterList()
        self.B_o = nn.ParameterList()
        for n_in, n_out in zip(self.architecture[:-1], self.architecture[1:]):
            shape = (n_in, n_out, self.SYNEs_per_synapse)
            self.S_min.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.S_max.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.V.append(nn.Parameter(torch.zeros(*shape, self.controls_per_device, device=DEVICE)))
            self.G_o.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
            self.B_o.append(nn.Parameter(torch.zeros(shape, device=DEVICE)))
        self._initialise()

    def train(self, mode: bool = True):
        super().train(mode)
        self.twin.eval()
        return self

    def squash_signal(self, raw):
        return 2.0 * torch.sigmoid(self.squash_gain * raw) - 1.0

    @staticmethod
    def squash_control(raw):
        return 2.0 * torch.sigmoid(raw) - 1.0

    def inverse_signal(self, value):
        probability = ((value + 1.0) * 0.5).clamp(1e-6, 1.0 - 1e-6)
        return torch.log(probability / (1.0 - probability)) / self.squash_gain

    def _initialise_ranges(self, layer, n_in, n_out):
        shape = (n_in, n_out, self.SYNEs_per_synapse)
        minimum_width = float(self.hp["init_dv_min"])
        maximum_width = min(float(self.hp["init_dv_max"]), 2.0 - 2.0 * self.physical_epsilon)
        count = int(np.prod(shape))

        widths = torch.linspace(minimum_width, maximum_width, count, device=DEVICE)
        step = (maximum_width - minimum_width) / float(max(count - 1, 1))
        jitter = (torch.rand(count, device=DEVICE) - 0.5) * step * 2.0 * float(self.hp["span_jitter"])
        widths = (widths + jitter).clamp(minimum_width, maximum_width)
        widths = widths[torch.randperm(count, device=DEVICE)].reshape(shape)
        fan = float(n_in * self.SYNEs_per_synapse)
        powerlaw_scale = max(0.1, min(1.0, fan ** (-float(self.hp["pl_beta"]))))
        widths = minimum_width + (widths - minimum_width) * powerlaw_scale

        inverted = torch.rand(shape, device=DEVICE) < float(self.hp["invert_probability"])
        centre_low = -1.0 + self.physical_epsilon + 0.5 * widths
        centre_high = 1.0 - self.physical_epsilon - 0.5 * widths
        centre = centre_low + torch.rand(shape, device=DEVICE) * (centre_high - centre_low)
        centre = torch.where(inverted, torch.zeros_like(centre), centre)
        physical_minimum = centre - 0.5 * widths
        physical_maximum = centre + 0.5 * widths
        uninverted_minimum = physical_minimum
        physical_minimum = torch.where(inverted, physical_maximum, physical_minimum)
        physical_maximum = torch.where(inverted, uninverted_minimum, physical_maximum)
        self.S_min[layer].copy_(self.inverse_signal(physical_minimum.clamp(-0.98, 0.98)))
        self.S_max[layer].copy_(self.inverse_signal(physical_maximum.clamp(-0.98, 0.98)))

    def _initialise(self):
        with torch.no_grad():
            for layer, (n_in, n_out) in enumerate(zip(self.architecture[:-1], self.architecture[1:])):
                self._initialise_ranges(layer, n_in, n_out)
                # pl_alpha applies independently to the two constant control voltages.
                shrink = float(n_in * self.SYNEs_per_synapse) ** (-float(self.hp["pl_alpha"]))
                limit = float(self.hp["control_init_scale"]) * math.sqrt(6.0 / 4.0) * shrink
                raw_minimum = math.log(0.4 / 0.6)
                self.V[layer].uniform_(-limit, limit)
                mask = self.V[layer] < raw_minimum
                while bool(mask.any()):
                    self.V[layer][mask] = torch.empty_like(self.V[layer][mask]).uniform_(raw_minimum, limit)
                    mask = self.V[layer] < raw_minimum
                self.G_o[layer].zero_()
                self.B_o[layer].zero_()

    def _twin_forward(self, flat_inputs):
        outputs = []
        for start in range(0, len(flat_inputs), TWIN_MICROBATCH):
            outputs.append(self.twin(flat_inputs[start:start + TWIN_MICROBATCH]).reshape(-1))
        return torch.cat(outputs, dim=0)

    def forward(self, inputs):
        activation = inputs
        for layer, (n_in, n_out) in enumerate(zip(self.architecture[:-1], self.architecture[1:])):
            scalar = activation.unsqueeze(2).unsqueeze(3)
            physical_minimum = self.squash_signal(self.S_min[layer])
            physical_maximum = self.squash_signal(self.S_max[layer])
            signal = (
                0.5 * (physical_maximum - physical_minimum).unsqueeze(0) * scalar
                + 0.5 * (physical_maximum + physical_minimum).unsqueeze(0)
            ).unsqueeze(-1)
            controls = self.squash_control(self.V[layer]).unsqueeze(0).expand(
                len(activation), -1, -1, -1, -1
            )
            twin_inputs = torch.cat([signal, controls], dim=-1)
            twin_outputs = self._twin_forward(twin_inputs.reshape(-1, 3)).reshape(
                len(activation), n_in, n_out, self.SYNEs_per_synapse
            )
            contribution = (
                self.output_gain_scale * self.G_o[layer].unsqueeze(0) * twin_outputs
                + self.B_o[layer].unsqueeze(0)
            )
            activation = contribution.sum(dim=(1, 3))
        return activation

    def parameter_groups(self):
        '''Three learning-rate groups: output gains and biases, controls, ranges.'''
        return [
            {"name": "gains_biases", "params": list(self.G_o) + list(self.B_o),
             "lr": float(self.hp["lr_gobo"]), "weight_decay": 0.0},
            {"name": "controls", "params": list(self.V),
             "lr": float(self.hp["lr_v"]), "weight_decay": 0.0},
            {"name": "ranges", "params": list(self.S_min) + list(self.S_max),
             "lr": float(self.hp["lr_s"]), "weight_decay": 0.0},
        ]

    def trainable_parameter_count(self):
        edges = sum(n_in * n_out for n_in, n_out in zip(self.architecture[:-1], self.architecture[1:]))
        receiving_neurons = sum(self.architecture[1:])
        return 5 * edges * self.SYNEs_per_synapse + receiving_neurons


def build_physical_kan(twin):
    return PhysicalKAN(twin, KAN_RECIPE["architecture"], KAN_RECIPE["SYNEs_per_synapse"], KAN_RECIPE).to(DEVICE)


if RUN_PHYSICAL_KAN:
    _probe = build_physical_kan(TWIN)
    print(
        "PREFLIGHT | [9,12,1] | 12 SYNEs/synapse | "
        f"{_probe.trainable_parameter_count():,} parameters"
    )
    del _probe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## ReLU MLP baselines

The baseline models are plain ReLU MLPs with `depth` hidden layers of the given width.

In [ ]:
class MLP(nn.Module):
    '''Plain ReLU MLP with default linear initialisation.'''

    def __init__(self, input_dim, width, depth):
        super().__init__()
        layers = []
        current = int(input_dim)
        for _ in range(int(depth)):
            layers.append(nn.Linear(current, int(width)))
            layers.append(nn.ReLU())
            current = int(width)
        layers.append(nn.Linear(current, 1))
        self.network = nn.Sequential(*layers)

    def forward(self, inputs):
        return self.network(inputs)

    def trainable_parameter_count(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


def mlp_parameter_count(width, depth, input_dim=9):
    total = input_dim * width + width
    for _ in range(depth - 1):
        total += width * width + width
    total += width + 1
    return int(total)


def build_mlp(recipe):
    return MLP(9, int(recipe["width"]), int(recipe["depth"])).to(DEVICE)


count_rows = []
if RUN_PHYSICAL_KAN:
    _kan = build_physical_kan(TWIN)
    count_rows.append({
        "model": "Physical KAN [9,12,1], 12 SYNEs/synapse",
        "parameter_count": _kan.trainable_parameter_count(),
    })
    del _kan
for label, recipe in MLP_RECIPES.items():
    count = mlp_parameter_count(recipe["width"], recipe["depth"])
    count_rows.append({
        "model": f"MLP {label}  {recipe['architecture']}",
        "parameter_count": count,
    })

PARAMETER_COUNTS = pd.DataFrame(count_rows)
PARAMETER_COUNTS

## Training



In [ ]:
# --------------------------------------------------------------------------- #
# Shared per-seed split, scalers and metrics
# --------------------------------------------------------------------------- #
def split_and_scale(seed: int) -> dict:
    '''RandomState(seed) reserves 20% as test; scalers are fitted on the full
    80% pool; 10% of the pool then becomes validation.'''
    seed = int(seed)
    rng = np.random.RandomState(seed)
    order = rng.permutation(len(X_ALL))
    train_pool_size = int(0.8 * len(X_ALL))
    pool_index, test_index = order[:train_pool_size], order[train_pool_size:]

    x_pool_raw = X_ALL[pool_index]
    y_pool_raw = Y_ALL[pool_index].reshape(-1, 1)
    x_test_raw = X_ALL[test_index]
    y_test_raw = Y_ALL[test_index].reshape(-1, 1)

    x_scaler = MinMaxScaler(feature_range=(-1.0, 1.0)).fit(x_pool_raw)
    y_scaler = MinMaxScaler(feature_range=(-1.0, 1.0)).fit(y_pool_raw)
    x_pool = torch.from_numpy(x_scaler.transform(x_pool_raw).astype(np.float32)).to(DEVICE)
    y_pool = torch.from_numpy(y_scaler.transform(y_pool_raw).astype(np.float32)).to(DEVICE)
    x_test = torch.from_numpy(x_scaler.transform(x_test_raw).astype(np.float32)).to(DEVICE)
    y_test = torch.from_numpy(y_scaler.transform(y_test_raw).astype(np.float32)).to(DEVICE)

    try:
        generator = torch.Generator(device=DEVICE)
    except TypeError:
        generator = torch.Generator()
    generator.manual_seed(seed)
    try:
        permutation = torch.randperm(len(x_pool), generator=generator, device=DEVICE)
    except RuntimeError:
        permutation = torch.randperm(len(x_pool), generator=generator).to(DEVICE)
    validation_size = int(max(1, round(0.10 * len(x_pool))))
    validation_local = permutation[:validation_size]
    train_local = permutation[validation_size:]

    return {
        "x_train": x_pool[train_local], "y_train": y_pool[train_local],
        "x_val": x_pool[validation_local], "y_val": y_pool[validation_local],
        "x_test": x_test, "y_test": y_test,
        "y_test_seconds": y_test_raw, "x_scaler": x_scaler, "y_scaler": y_scaler,
    }


@torch.no_grad()
def predict_chunks(model, inputs, chunk_size=8192):
    was_training = model.training
    model.eval()
    amp = bool(USE_AMP and DEVICE.type == "cuda")
    outputs = []
    for start in range(0, len(inputs), int(chunk_size)):
        with autocast_context(amp):
            outputs.append(model(inputs[start:start + int(chunk_size)]).float())
    if was_training:
        model.train()
    return torch.cat(outputs, dim=0)


def metric_bundle(y_scaled, prediction_scaled, y_scaler) -> dict:
    y_scaled = np.asarray(y_scaled, dtype=np.float64).reshape(-1, 1)
    prediction_scaled = np.asarray(prediction_scaled, dtype=np.float64).reshape(-1, 1)
    y_seconds = y_scaler.inverse_transform(y_scaled).ravel()
    prediction_seconds = y_scaler.inverse_transform(prediction_scaled).ravel()
    mse_seconds = float(mean_squared_error(y_seconds, prediction_seconds))
    variance = float(np.var(y_seconds))
    return {
        "test_nMSE": float(mean_squared_error(y_scaled, prediction_scaled)),
        "test_variance_nMSE": float(mse_seconds / variance) if variance > 0 else float("nan"),
        "test_RMSE_seconds": float(math.sqrt(mse_seconds)),
        "test_MAE_seconds": float(mean_absolute_error(y_seconds, prediction_seconds)),
        "test_R2": float(r2_score(y_seconds, prediction_seconds)),
    }


# --------------------------------------------------------------------------- #
# Physical KAN training: warm-start selection and continuation
# --------------------------------------------------------------------------- #
def kan_compact_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items() if not k.startswith("twin.")}


def kan_make_optimizer(model):
    return torch.optim.Adam(
        model.parameter_groups(),
        betas=(float(KAN_RECIPE["beta1"]), float(KAN_RECIPE["beta2"])),
        eps=float(KAN_RECIPE["adam_eps"]),
    )


def kan_cosine_scale(epoch):
    progress = (int(epoch) - 1) / float(max(KAN_MAX_EPOCHS - 1, 1))
    minimum = float(KAN_RECIPE["lr_min_frac"])
    return minimum + 0.5 * (1.0 - minimum) * (1.0 + math.cos(math.pi * progress))


def kan_set_learning_rates(optimizer, epoch):
    scale = kan_cosine_scale(epoch)
    base = [float(KAN_RECIPE["lr_gobo"]), float(KAN_RECIPE["lr_v"]), float(KAN_RECIPE["lr_s"])]
    for group, base_rate in zip(optimizer.param_groups, base):
        group["lr"] = base_rate * scale


def kan_validation_mse(model, split):
    return float(F.mse_loss(predict_chunks(model, split["x_val"], chunk_size=2048), split["y_val"]).item())


def kan_train_one_epoch(model, optimizer, scaler, split, epoch):
    kan_set_learning_rates(optimizer, epoch)
    model.train()
    order = torch.randperm(len(split["x_train"]), device=DEVICE)
    running_loss, seen = 0.0, 0
    amp = bool(USE_AMP and DEVICE.type == "cuda")
    batch_size = int(KAN_RECIPE["batch_size"])
    for start in range(0, len(order), batch_size):
        index = order[start:start + batch_size]
        x_batch, y_batch = split["x_train"][index], split["y_train"][index]
        optimizer.zero_grad(set_to_none=True)
        with autocast_context(amp):
            prediction = model(x_batch)
            loss = F.mse_loss(prediction, y_batch)
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f"Non-finite KAN training loss at epoch {epoch}")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], float(KAN_RECIPE["grad_clip"]))
        scaler.step(optimizer)
        scaler.update()
        running_loss += float(loss.detach()) * len(index)
        seen += len(index)
    return running_loss / float(max(seen, 1))


def kan_select_initialisation(twin, split, run_seed, verbose):
    best = None
    for init_index in range(KAN_WARMUP_INITIALISATIONS):
        init_seed = int(run_seed) * 100 + init_index
        seed_everything(init_seed)
        model = build_physical_kan(twin)
        optimizer = kan_make_optimizer(model)
        scaler = make_grad_scaler(USE_AMP and DEVICE.type == "cuda")
        train_reference, last_improvement, history = float("inf"), 0, []
        train_mse = float("nan")
        for epoch in range(1, KAN_WARMUP_EPOCHS + 1):
            train_mse = kan_train_one_epoch(model, optimizer, scaler, split, epoch)
            history.append({"epoch": epoch, "train_mse": float(train_mse), "validation_mse": np.nan})
            if train_mse < train_reference - KAN_EARLY_STOP_MIN_DELTA:
                train_reference, last_improvement = float(train_mse), epoch
        val_mse = kan_validation_mse(model, split)
        history[-1]["validation_mse"] = float(val_mse)
        state = kan_compact_state(model)
        if verbose:
            print(f"  warm start {init_index + 1:02d}/{KAN_WARMUP_INITIALISATIONS} seed={init_seed} | "
                  f"train={train_mse:.6e} | val={val_mse:.6e}", flush=True)
        runtime = {
            "model": model, "optimizer": optimizer, "scaler": scaler,
            "current_epoch": KAN_WARMUP_EPOCHS, "best_validation": float(val_mse),
            "best_epoch": KAN_WARMUP_EPOCHS, "best_state": state,
            "train_reference": train_reference, "last_train_improvement_epoch": last_improvement,
            "init_index": init_index, "init_seed": init_seed, "history": history,
            "early_stopped": False,
        }
        if best is None or runtime["best_validation"] < best["best_validation"]:
            best = runtime
        else:
            del runtime["model"], runtime["optimizer"], runtime["scaler"]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if best is None:
        raise RuntimeError("No finite warm-start initialisation survived")
    if verbose:
        print(f"  selected warm start {best['init_index']} (seed {best['init_seed']}) | "
              f"epoch-{KAN_WARMUP_EPOCHS} val={best['best_validation']:.6e}", flush=True)
    return best


def kan_update_best_checkpoint(runtime, validation, epoch):
    improved = validation < runtime["best_validation"] - 1e-12
    if improved:
        runtime["best_validation"], runtime["best_epoch"] = float(validation), int(epoch)
        runtime["best_state"] = kan_compact_state(runtime["model"])
    return improved


def kan_continue_training(runtime, split, seed, verbose):
    seed_everything(runtime["init_seed"] + 10_000_000 + runtime["current_epoch"])
    started = time.time()
    for epoch in range(runtime["current_epoch"] + 1, KAN_MAX_EPOCHS + 1):
        train_mse = kan_train_one_epoch(runtime["model"], runtime["optimizer"], runtime["scaler"], split, epoch)
        runtime["current_epoch"] = epoch
        if train_mse < runtime["train_reference"] - KAN_EARLY_STOP_MIN_DELTA:
            runtime["train_reference"], runtime["last_train_improvement_epoch"] = float(train_mse), epoch
        evaluate = epoch % KAN_EVALUATE_EVERY == 0 or epoch == KAN_MAX_EPOCHS
        val_mse, improved = np.nan, False
        if evaluate:
            val_mse = kan_validation_mse(runtime["model"], split)
            improved = kan_update_best_checkpoint(runtime, val_mse, epoch)
        runtime["history"].append({"epoch": epoch, "train_mse": float(train_mse), "validation_mse": float(val_mse)})
        if evaluate and verbose:
            print(f"  [seed {seed}] epoch {epoch:4d}/{KAN_MAX_EPOCHS} | train={train_mse:.6e} | "
                  f"val={val_mse:.6e} | best={runtime['best_validation']:.6e} @ {runtime['best_epoch']:4d}"
                  f"{' *' if improved else ''} | {(time.time() - started) / 60.0:.1f} min", flush=True)
        if (epoch >= KAN_EARLY_STOP_MIN_EPOCH
                and epoch - runtime["last_train_improvement_epoch"] >= KAN_EARLY_STOP_PATIENCE):
            runtime["early_stopped"] = True
            if verbose:
                print(f"  EARLY STOP at epoch {epoch}; best validation {runtime['best_validation']:.6e} "
                      f"@ {runtime['best_epoch']}", flush=True)
            break
    return runtime


def train_kan_seed(twin, seed, verbose=False):
    split = split_and_scale(seed)
    runtime = kan_select_initialisation(twin, split, run_seed=seed, verbose=verbose)
    runtime = kan_continue_training(runtime, split, seed, verbose=verbose)
    runtime["model"].load_state_dict(runtime["best_state"], strict=False)
    prediction = predict_chunks(runtime["model"], split["x_test"], chunk_size=8192)
    metrics = metric_bundle(split["y_test"].detach().cpu().numpy(), prediction.detach().cpu().numpy(), split["y_scaler"])
    extra = {
        "best_epoch": int(runtime["best_epoch"]),
        "epochs_ran": int(runtime["current_epoch"]), "early_stopped": bool(runtime["early_stopped"]),
        "selected_init_seed": int(runtime["init_seed"]),
    }
    test_prediction_seconds = split["y_scaler"].inverse_transform(
        prediction.detach().cpu().numpy().reshape(-1, 1)).ravel()
    prediction_bundle = {
        "target_seconds": split["y_test_seconds"].reshape(-1),
        "prediction_seconds": test_prediction_seconds,
    }
    model = runtime["model"]
    del runtime
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, metrics, extra, prediction_bundle


# --------------------------------------------------------------------------- #
# MLP training: fixed epoch budget, batch 2048, gradient accumulation
# --------------------------------------------------------------------------- #
def mlp_build_optimizer(model, recipe):
    return torch.optim.Adam(
        model.parameters(),
        lr=float(recipe["lr"]),
        betas=(float(recipe["beta1"]), float(recipe["beta2"])),
        eps=float(recipe["adam_eps"]),
        weight_decay=float(recipe["weight_decay"]),
    )


def mlp_build_scheduler(optimizer, recipe):
    if str(recipe["scheduler"]) != "cosine":
        return None
    eta_min = float(recipe["lr"]) * float(recipe["final_lr_fraction"])
    return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=int(recipe["cosine_t_max"]), eta_min=eta_min)


def mlp_compact_state(model):
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def mlp_validation_mse(model, split):
    return float(F.mse_loss(predict_chunks(model, split["x_val"], chunk_size=2048), split["y_val"]).item())


def mlp_train_one_epoch(model, optimizer, scaler, scheduler, split, recipe, epoch):
    model.train()
    amp = bool(USE_AMP and DEVICE.type == "cuda")
    batch_size = int(recipe["batch_size"])
    accumulation_steps = int(recipe["gradient_accumulation_steps"])
    permutation = torch.randperm(len(split["x_train"]), device=DEVICE)
    optimizer.zero_grad(set_to_none=True)
    number_of_batches = int(math.ceil(len(permutation) / batch_size))
    running_loss, seen = 0.0, 0
    for batch_number, start in enumerate(range(0, len(permutation), batch_size), 1):
        index = permutation[start:start + batch_size]
        x_batch, y_batch = split["x_train"][index], split["y_train"][index]
        # Weight each minibatch by its share of the current accumulation group,
        # so a short final group is not underweighted.
        zero_based = batch_number - 1
        group_first_batch = (zero_based // accumulation_steps) * accumulation_steps
        group_last_batch = min(group_first_batch + accumulation_steps, number_of_batches)
        group_first_sample = group_first_batch * batch_size
        group_last_sample = min(group_last_batch * batch_size, len(permutation))
        loss_weight = float(len(index)) / float(max(1, group_last_sample - group_first_sample))
        with autocast_context(amp):
            prediction = model(x_batch)
            batch_mse = F.mse_loss(prediction, y_batch)
            loss = batch_mse * loss_weight
        if not bool(torch.isfinite(loss)):
            raise FloatingPointError(f"Non-finite MLP training loss at epoch {epoch}")
        scaler.scale(loss).backward()
        if batch_number % accumulation_steps == 0 or batch_number == number_of_batches:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
        running_loss += float(batch_mse.detach()) * len(index)
        seen += len(index)
    if scheduler is not None:
        scheduler.step()
    return running_loss / float(max(seen, 1))


def mlp_select_initialisation(recipe, split, run_seed, verbose):
    best = None
    for init_index in range(KAN_WARMUP_INITIALISATIONS):
        init_seed = int(run_seed) * 100 + init_index
        seed_everything(init_seed)
        model = build_mlp(recipe)
        optimizer = mlp_build_optimizer(model, recipe)
        scheduler = mlp_build_scheduler(optimizer, recipe)
        scaler = make_grad_scaler(USE_AMP and DEVICE.type == "cuda")
        train_reference, last_improvement, history = float("inf"), 0, []
        train_mse = float("nan")
        for epoch in range(1, KAN_WARMUP_EPOCHS + 1):
            train_mse = mlp_train_one_epoch(model, optimizer, scaler, scheduler, split, recipe, epoch)
            history.append({"epoch": epoch, "train_mse": float(train_mse), "validation_mse": np.nan})
            if train_mse < train_reference - KAN_EARLY_STOP_MIN_DELTA:
                train_reference, last_improvement = float(train_mse), epoch
        val_mse = mlp_validation_mse(model, split)
        history[-1]["validation_mse"] = float(val_mse)
        state = mlp_compact_state(model)
        if verbose:
            print(f"  warm start {init_index + 1:02d}/{KAN_WARMUP_INITIALISATIONS} seed={init_seed} | "
                  f"train={train_mse:.6e} | val={val_mse:.6e}", flush=True)
        runtime = {
            "model": model, "optimizer": optimizer, "scheduler": scheduler, "scaler": scaler,
            "current_epoch": KAN_WARMUP_EPOCHS, "best_validation": float(val_mse),
            "best_epoch": KAN_WARMUP_EPOCHS, "best_state": state,
            "train_reference": train_reference, "last_train_improvement_epoch": last_improvement,
            "init_index": init_index, "init_seed": init_seed, "history": history,
            "early_stopped": False,
        }
        if best is None or runtime["best_validation"] < best["best_validation"]:
            best = runtime
        else:
            del runtime["model"], runtime["optimizer"], runtime["scheduler"], runtime["scaler"]
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if best is None:
        raise RuntimeError("No finite warm-start initialisation survived")
    if verbose:
        print(f"  selected warm start {best['init_index']} (seed {best['init_seed']}) | "
              f"epoch-{KAN_WARMUP_EPOCHS} val={best['best_validation']:.6e}", flush=True)
    return best


def mlp_update_best_checkpoint(runtime, validation, epoch):
    improved = validation < runtime["best_validation"] - 1e-12
    if improved:
        runtime["best_validation"], runtime["best_epoch"] = float(validation), int(epoch)
        runtime["best_state"] = mlp_compact_state(runtime["model"])
    return improved


def mlp_continue_training(runtime, split, seed, recipe, verbose):
    seed_everything(runtime["init_seed"] + 10_000_000 + runtime["current_epoch"])
    started = time.time()
    for epoch in range(runtime["current_epoch"] + 1, MLP_MAX_EPOCHS + 1):
        train_mse = mlp_train_one_epoch(runtime["model"], runtime["optimizer"], runtime["scaler"],
                                        runtime["scheduler"], split, recipe, epoch)
        runtime["current_epoch"] = epoch
        if train_mse < runtime["train_reference"] - KAN_EARLY_STOP_MIN_DELTA:
            runtime["train_reference"], runtime["last_train_improvement_epoch"] = float(train_mse), epoch
        evaluate = epoch % KAN_EVALUATE_EVERY == 0 or epoch == MLP_MAX_EPOCHS
        val_mse, improved = np.nan, False
        if evaluate:
            val_mse = mlp_validation_mse(runtime["model"], split)
            improved = mlp_update_best_checkpoint(runtime, val_mse, epoch)
        runtime["history"].append({"epoch": epoch, "train_mse": float(train_mse), "validation_mse": float(val_mse)})
        if evaluate and verbose:
            print(f"  [{recipe['label']} seed {seed}] epoch {epoch:4d}/{MLP_MAX_EPOCHS} | train={train_mse:.6e} | "
                  f"val={val_mse:.6e} | best={runtime['best_validation']:.6e} @ {runtime['best_epoch']:4d}"
                  f"{' *' if improved else ''} | {(time.time() - started) / 60.0:.1f} min", flush=True)
        if (epoch >= KAN_EARLY_STOP_MIN_EPOCH
                and epoch - runtime["last_train_improvement_epoch"] >= KAN_EARLY_STOP_PATIENCE):
            runtime["early_stopped"] = True
            if verbose:
                print(f"  EARLY STOP at epoch {epoch}; best validation {runtime['best_validation']:.6e} "
                      f"@ {runtime['best_epoch']}", flush=True)
            break
    return runtime


def train_mlp_fixed(recipe, seed, verbose=False):
    split = split_and_scale(seed)
    runtime = mlp_select_initialisation(recipe, split, run_seed=seed, verbose=verbose)
    runtime = mlp_continue_training(runtime, split, seed, recipe, verbose=verbose)
    runtime["model"].load_state_dict(runtime["best_state"], strict=True)
    prediction = predict_chunks(runtime["model"], split["x_test"], chunk_size=8192)
    metrics = metric_bundle(split["y_test"].detach().cpu().numpy(), prediction.detach().cpu().numpy(), split["y_scaler"])
    extra = {
        "best_epoch": int(runtime["best_epoch"]),
        "epochs_ran": int(runtime["current_epoch"]), "early_stopped": bool(runtime["early_stopped"]),
        "selected_init_seed": int(runtime["init_seed"]),
    }
    test_prediction_seconds = split["y_scaler"].inverse_transform(
        prediction.detach().cpu().numpy().reshape(-1, 1)).ravel()
    prediction_bundle = {
        "target_seconds": split["y_test_seconds"].reshape(-1),
        "prediction_seconds": test_prediction_seconds,
    }
    model = runtime["model"]
    del runtime
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return model, metrics, extra, prediction_bundle


## Result logging

Each run ID contains the model family, configuration, recipe signature, seed and epoch budget. Completed rows are stored in `runs.csv`.

In [ ]:
RESULTS_PATH = OUTPUT_ROOT / "runs.csv"
REPRESENTATIVE_SEED = SEEDS[0]


def load_results() -> pd.DataFrame:
    if not RESULTS_PATH.exists():
        return pd.DataFrame()
    frame = pd.read_csv(RESULTS_PATH)
    if "test_nMSE" not in frame.columns:
        candidates = [
            column for column in frame.columns
            if column.startswith("test_")
            and column.endswith("nMSE")
            and column != "test_variance_nMSE"
        ]
        if len(candidates) == 1:
            frame = frame.rename(columns={candidates[0]: "test_nMSE"})
    if "run_id" not in frame:
        raise ValueError(f"Existing results file has no run_id column: {RESULTS_PATH}")
    return frame.drop_duplicates("run_id", keep="last").reset_index(drop=True)


def save_results(frame: pd.DataFrame) -> None:
    temporary = RESULTS_PATH.with_suffix(".tmp")
    frame.to_csv(temporary, index=False)
    temporary.replace(RESULTS_PATH)


def prediction_path(model_family, configuration, seed, signature):
    return PREDICTION_ROOT / f"{model_family}_{configuration}_seed{seed}_{signature}.npz"


RESULTS = load_results()
COMPLETED_RUN_IDS = set(RESULTS["run_id"].astype(str)) if len(RESULTS) else set()


def run_and_record(*, model_family, configuration, seed, recipe, max_epochs,
                   parameter_count, save_prediction=False, verbose=False) -> bool:
    global RESULTS

    signature = recipe_signature(recipe)
    run_id = "|".join(map(str, [model_family, configuration, seed, signature, max_epochs]))
    if run_id in COMPLETED_RUN_IDS:
        return False

    started = time.time()
    if model_family == "KAN":
        model, metrics, extra, prediction = train_kan_seed(TWIN, seed, verbose=verbose)
    elif model_family == "MLP":
        model, metrics, extra, prediction = train_mlp_fixed(recipe, seed, verbose=verbose)
    else:
        raise ValueError(f"Unknown model family: {model_family}")

    row = {
        "run_id": run_id, "recipe_signature": signature,
        "model_family": model_family, "configuration": configuration, "seed": int(seed),
        "parameter_count": int(parameter_count), "max_epochs": int(max_epochs),
        "elapsed_minutes": float((time.time() - started) / 60.0),
    }
    row.update({k: v for k, v in metrics.items()})
    row.update({k: v for k, v in extra.items()})
    if save_prediction:
        np.savez_compressed(
            prediction_path(model_family, configuration, seed, signature),
            target_seconds=prediction["target_seconds"],
            prediction_seconds=prediction["prediction_seconds"],
        )

    RESULTS = pd.concat([RESULTS, pd.DataFrame([row])], ignore_index=True)
    COMPLETED_RUN_IDS.add(run_id)
    save_results(RESULTS)
    print(f"{model_family:3s} | {configuration:9s} | seed {seed} | "
          f"test nMSE {metrics['test_nMSE']:.4e} | test R2 {metrics['test_R2']:.4f}")
    del model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return True

## Run the physical SYNE-KAN

The `[9, 12, 1]` physical SYNE-KAN is trained once per seed.

In [ ]:
if RUN_PHYSICAL_KAN:
    parameter_count = build_physical_kan(TWIN).trainable_parameter_count()
    for seed in SEEDS:
        run_and_record(
            model_family="KAN",
            configuration="9_12_1",
            seed=seed,
            recipe=KAN_RECIPE,
            max_epochs=KAN_MAX_EPOCHS,
            parameter_count=parameter_count,
            save_prediction=(seed == REPRESENTATIVE_SEED),
            verbose=True,
        )

print(f"Stored {len(RESULTS)} rows in {RESULTS_PATH}")

## Run the MLP baselines

Each MLP configuration runs over every seed with its own hyperparam set. 

In [ ]:
if RUN_MLP_BASELINES:
    for label, recipe in MLP_RECIPES.items():
        parameter_count = mlp_parameter_count(recipe["width"], recipe["depth"])
        for seed in SEEDS:
            run_and_record(
                model_family="MLP",
                configuration=label,
                seed=seed,
                recipe=recipe,
                max_epochs=MLP_MAX_EPOCHS,
                parameter_count=parameter_count,
                save_prediction=(seed == REPRESENTATIVE_SEED),
                verbose=False,
            )

print(f"Stored {len(RESULTS)} rows in {RESULTS_PATH}")

## Results and comparison

Test nMSE is aggregated over seeds for each configuration.

In [ ]:
RESULTS = load_results()
if RESULTS.empty:
    raise RuntimeError("No run data found. Run a KAN or MLP sweep first.")

summary = (
    RESULTS.groupby(["model_family", "configuration", "parameter_count"], dropna=False)
    .agg(
        seeds=("seed", "nunique"),
        **{
            "mean test nMSE": ("test_nMSE", "mean"),
            "std test nMSE": ("test_nMSE", lambda v: float(np.std(v, ddof=0))),
        },
        mean_test_R2=("test_R2", "mean"),
        mean_test_RMSE_seconds=("test_RMSE_seconds", "mean"),
    )
    .reset_index()
    .sort_values("parameter_count")
)
summary.to_csv(OUTPUT_ROOT / "summary_by_configuration.csv", index=False)


def make_comparison_plot(results, summary):
    fig, ax = plt.subplots(figsize=(7.4, 5.2))
    colours = {"9_12_1": KAN_COLOUR, "d5_w300": MLP_COLOURS["d5_w300"], "d2_w50": MLP_COLOURS["d2_w50"]}
    display = {"9_12_1": "Physical KAN [9,12,1]", "d5_w300": "MLP d5_w300", "d2_w50": "MLP d2_w50"}

    for _, row in summary.iterrows():
        configuration = row["configuration"]
        colour = colours.get(configuration, "#555555")
        seed_values = results.loc[results["configuration"] == configuration, "test_nMSE"].to_numpy(float)
        ax.scatter(np.full_like(seed_values, row["parameter_count"]), seed_values,
                   s=18, color=colour, alpha=0.35, zorder=2)
        ax.errorbar(row["parameter_count"], row["mean test nMSE"],
                    yerr=row["std test nMSE"], fmt="o", ms=9, color=colour,
                    capsize=4, lw=1.6, zorder=3, label=display.get(configuration, configuration))
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel("Parameter count")
    ax.set_ylabel("Test nMSE")
    ax.set_title("NASA battery remaining-time regression")
    ax.grid(alpha=GRID_ALPHA, which="both")
    ax.legend(frameon=False, loc="best")
    fig.tight_layout()
    save_figure(fig, OUTPUT_ROOT / "parameter_performance_comparison.png")
    return fig


comparison_figure = make_comparison_plot(RESULTS, summary)
plt.show()
summary

## Outputs

Predicted versus true remaining discharge time on the held-out test set.

In [ ]:
def find_prediction(model_family, configuration, seed):
    matches = sorted(PREDICTION_ROOT.glob(f"{model_family}_{configuration}_seed{seed}_*.npz"))
    return matches[-1] if matches else None


def plot_parity(panels):
    available = [(family, configuration, title) for family, configuration, title in panels
                 if find_prediction(family, configuration, REPRESENTATIVE_SEED) is not None]
    if not available:
        print("No saved predictions yet; run the sweeps with the representative seed first.")
        return None

    fig, axes = plt.subplots(1, len(available), figsize=(4.4 * len(available), 4.2))
    axes = np.atleast_1d(axes).reshape(-1)
    colours = {"KAN": KAN_COLOUR, "MLP": MLP_COLOURS["d5_w300"]}
    for ax, (family, configuration, title) in zip(axes, available):
        reference = np.load(find_prediction(family, configuration, REPRESENTATIVE_SEED))
        target = reference["target_seconds"] / 3600.0
        prediction = reference["prediction_seconds"] / 3600.0
        limit = float(max(target.max(), prediction.max()))
        ax.plot([0, limit], [0, limit], "--", color=TARGET_COLOUR, lw=1.2)
        ax.scatter(target, prediction, s=6, alpha=0.25,
                   color=colours.get(family, MLP_COLOURS["d2_w50"]))
        ax.set_xlabel("True remaining time (hours)")
        ax.set_ylabel("Predicted remaining time (hours)")
        ax.set_title(title)
        ax.grid(alpha=GRID_ALPHA)
        ax.set_aspect("equal", adjustable="box")
    fig.tight_layout()
    save_figure(fig, OUTPUT_ROOT / "representative_parity.png")
    return fig


parity_figure = plot_parity([
    ("KAN", "9_12_1", "Physical KAN [9,12,1]"),
    ("MLP", "d5_w300", "MLP d5_w300"),
    ("MLP", "d2_w50", "MLP d2_w50"),
])
plt.show()

## Output files

Each profile writes:

- `runs.csv`
- `summary_by_configuration.csv`
- `fixed_release_recipes.json`
- `dataset_<signature>.npz`
- `predictions/*.npz`
- `parameter_performance_comparison.png`
- `representative_parity.png`